# Module 26 — Exercise 3: Resilient Retry Transport with Backoff

In this exercise, you will design a custom retry strategy for transient HTTP errors and rate limits (429, 502, 503, 504).

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 26 README, Module 16 |



## 1. Retry Calculations: Exponential Backoff with Jitter

Under concurrency, retrying at exact intervals causes the 'thundering herd' problem. Adding jitter (random noise) spreads out load.


In [ ]:
import random

def calculate_delay(attempt: int, base_delay: float = 0.5, max_delay: float = 30.0) -> float:
    """Calculate exponential delay with full jitter."""
    backoff = min(max_delay, base_delay * (2 ** attempt))
    return random.uniform(0, backoff)

for i in range(1, 4):
    print(f"Attempt {i} sampled delay: {calculate_delay(i):.2f}s")



# Your turn


### Task 1: Implement `should_retry(status_code)`

Return `True` if `status_code` is 429 (Too Many Requests) or a transient 5xx error (502, 503, 504). Return `False` for client errors like 400, 401, 403, 404, or standard 200.


In [ ]:
# ANSWER 1
def should_retry(status_code: int) -> bool:
    return status_code in (429, 502, 503, 504)



### Task 2: Resilient Requester

Implement `resilient_get(client, url, max_retries, sleep_func)` which attempts to fetch `url`. If `should_retry` is true or a `ConnectTimeout` occurs, it retries up to `max_retries` times using `sleep_func(delay)`. If retries are exhausted, it returns the error response or raises the exception.


In [ ]:
# ANSWER 2
import httpx

def resilient_get(client, url: str, max_retries: int = 3, sleep_func=lambda d: None):
    last_resp = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.get(url)
            last_resp = resp
            if not should_retry(resp.status_code):
                return resp
            delay = 0.1 * (2 ** attempt)
            sleep_func(delay)
        except (httpx.ConnectTimeout, httpx.ReadTimeout):
            if attempt == max_retries:
                raise
            delay = 0.1 * (2 ** attempt)
            sleep_func(delay)
    return last_resp



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

class FakeRetryClient:
    def __init__(self, codes):
        self.codes = list(codes)
        self.calls = 0
    def get(self, url):
        self.calls += 1
        code = self.codes.pop(0) if self.codes else 200
        return MockResponse(code, {"status": code})

c1 = FakeRetryClient([503, 503, 200])
r1 = resilient_get(c1, "https://api.test", max_retries=3)

c2 = FakeRetryClient([404])
r2 = resilient_get(c2, "https://api.test", max_retries=3)

results = [
    check(should_retry(429) is True, "Task 1: 429 is retryable"),
    check(should_retry(503) is True, "Task 1: 503 is retryable"),
    check(should_retry(404) is False, "Task 1: 404 is not retryable"),
    check(c1.calls == 3 and r1.status_code == 200, "Task 2: retried 503 twice and succeeded on 3rd attempt"),
    check(c2.calls == 1 and r2.status_code == 404, "Task 2: 404 does not retry"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

